# 4. Prepare motion windows and spatial features

For each AVI in `data/raw/videos/`, run VideoMeta motion gating and object tracking. It writes a `motion_scores.png` graph, annotated window MP4s, spatial feature files, and `prepared_windows.json` to `data/interim/<video-name>/`.

Run this notebook with the VideoMeta vision dependencies installed.

In [2]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
from videometa import (
    DetectionConfig,
    MotionGateConfig,
    ObjectBoundaryExtractor,
    RelevantWindowFinder,
    WindowSpatialFeatureJoiner,
    resolve_video_source,
)

PROJECT_ROOT = next(
    parent
    for parent in (Path.cwd(), *Path.cwd().parents)
    if (parent / "notebooks").is_dir() and (parent / "annotation_pipeline").is_dir()
)
RAW_VIDEOS = PROJECT_ROOT / "data" / "raw" / "videos"
INTERIM_DIR = PROJECT_ROOT / "data" / "interim"
MAX_VIDEOS = None  # Set an integer for a smaller experiment.
MAX_WINDOWS = None  # Set an integer to keep only the highest-motion windows.

motion_config = MotionGateConfig(
    window_seconds=12,
    stride_seconds=10,
    motion_threshold=0.015,
    warmup_seconds=2,
    mog_history=100,
)
window_finder = RelevantWindowFinder(motion_config)
boundary_extractor = ObjectBoundaryExtractor(
    DetectionConfig(model_path="yolo26n.pt", confidence_threshold=0.25, spatial_grid=3)
)

video_paths = sorted(RAW_VIDEOS.glob("*.avi"))[:MAX_VIDEOS]
if not video_paths:
    raise FileNotFoundError(f"No AVI files found in {RAW_VIDEOS}")

for video_path in video_paths:
    video_name = video_path.stem.removesuffix(".r13")
    output_dir = INTERIM_DIR / video_name
    frames_dir = output_dir / "window_frames"
    output_dir.mkdir(parents=True, exist_ok=True)

    source_path = resolve_video_source(video_path)
    video_info, motion_samples = window_finder.sample_motion(source_path)
    windows = window_finder.build_windows(motion_samples, video_info.duration_seconds)
    selected_windows = sorted(
        (window for window in windows if window.is_relevant),
        key=lambda window: window.peak_motion,
        reverse=True,
    )[:MAX_WINDOWS]

    figure, axis = plt.subplots(figsize=(14, 4))
    axis.plot(
        [sample.timestamp_seconds for sample in motion_samples],
        [sample.score for sample in motion_samples],
        linewidth=0.75,
    )
    axis.axhline(motion_config.motion_threshold, color="crimson", linestyle="--", label="threshold")
    for window in selected_windows:
        axis.axvspan(window.start_seconds, window.end_seconds, color="seagreen", alpha=0.15)
    axis.set(title=f"Motion scores: {video_name}", xlabel="Time (seconds)", ylabel="Motion score")
    axis.legend()
    figure.savefig(output_dir / "motion_scores.png", dpi=150, bbox_inches="tight")
    plt.close(figure)

    object_annotations = boundary_extractor.extract(source_path, selected_windows, video_info.fps)

    joiner = WindowSpatialFeatureJoiner(
        output_directory=frames_dir,
        annotated_video_size=(640, 360),
        annotated_video_fps=4,
        annotated_video_max_frames=32,
        encode_frame_images=False,
    )
    prepared_windows = [joiner.prepare(source_path, annotation) for annotation in object_annotations]
    manifest = [
        {
            "start_seconds": item.start_seconds,
            "end_seconds": item.end_seconds,
            "object_features": item.object_features,
            "frame_features": item.frame_features,
            "image_messages": item.image_messages,
            "artifact_directory": item.artifact_directory,
            "annotated_video_path": item.annotated_video_path,
        }
        for item in prepared_windows
    ]
    (output_dir / "prepared_windows.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
    print(f"Prepared {len(prepared_windows)} windows for {video_name}")

Prepared 17 windows for 2018-03-05.13-15-00.13-20-00.bus.G340
Prepared 0 windows for 2018-03-05.13-15-00.13-20-00.bus.G505
Prepared 2 windows for 2018-03-05.13-15-00.13-20-00.bus.G506
Prepared 12 windows for 2018-03-05.13-15-00.13-20-00.hospital.G341
Prepared 0 windows for 2018-03-05.13-15-00.13-20-00.school.G423
Prepared 10 windows for 2018-03-05.13-15-00.13-20-00.school.G424
Prepared 0 windows for 2018-03-05.13-15-01.13-20-01.school.G328
Prepared 7 windows for 2018-03-05.13-20-00.13-25-00.bus.G340
Prepared 3 windows for 2018-03-05.13-20-00.13-25-00.school.G336
Prepared 0 windows for 2018-03-05.13-20-00.13-25-00.school.G423
